# Road Surface Condition Detection Using CNN

Classifies a road photo as **normal** or **pothole**, and shows *where* the model looked using **Grad-CAM**.

We train and compare two models:
- **Model A** – a custom CNN trained from scratch
- **Model B** – MobileNetV2 transfer learning (ImageNet weights) + fine-tuning

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
At the end, the trained models and `classes.json` are downloaded to your computer — put them in the repo's `models/` folder.

## 1. Setup
Imports, random seeds for reproducibility, and a GPU check.

In [ ]:
# kagglehub is preinstalled on Colab; this makes sure it's there (quiet install)
%pip install -q kagglehub

import os, json, random, shutil, pathlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

SEED = 42
keras.utils.set_random_seed(SEED)   # seeds Python, NumPy and TensorFlow together

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Chart colours used throughout (series 1 = blue, series 2 = orange)
C1, C2, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#898781", "#e1e0d9"
plt.rcParams.update({"axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
                     "axes.spines.top": False, "axes.spines.right": False})

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", bool(gpus), gpus)
if not gpus:
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU (training will still work on CPU, just slower).")

## 2. Dataset
Download the Kaggle **Pothole Detection Dataset** (`atulyakumar98/pothole-detection-dataset`) with `kagglehub`,
auto-detect the folder that contains one sub-folder per class, copy it to a working folder, and remove any images TensorFlow can't decode.

In [ ]:
import kagglehub

download_path = kagglehub.dataset_download("atulyakumar98/pothole-detection-dataset")
print("Downloaded to:", download_path)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}   # formats image_dataset_from_directory accepts

def has_images(folder):
    return any(p.suffix.lower() in IMG_EXTS for p in pathlib.Path(folder).iterdir() if p.is_file())

def find_class_root(root):
    """Return the shallowest folder whose sub-folders (>=2) all contain images = one sub-folder per class."""
    for dirpath, dirnames, _ in sorted(os.walk(root), key=lambda t: t[0].count(os.sep)):
        subdirs = [os.path.join(dirpath, d) for d in dirnames]
        if len(subdirs) >= 2 and all(has_images(d) for d in subdirs):
            return pathlib.Path(dirpath)
    raise RuntimeError("Could not find a folder with one sub-folder per class.")

class_root = find_class_root(download_path)
print("Class folder found:", class_root)

# Copy to a writable working folder (keeps the kagglehub cache untouched)
DATA_DIR = pathlib.Path("/content/data/road_images") if os.path.exists("/content") else pathlib.Path("data/road_images")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
shutil.copytree(class_root, DATA_DIR)

def count_images(root):
    return {d.name: sum(1 for f in d.iterdir() if f.suffix.lower() in IMG_EXTS) for d in sorted(root.iterdir()) if d.is_dir()}

print("Images per class:", count_images(DATA_DIR))

In [ ]:
# Remove files TensorFlow can't decode (corrupt images, wrong formats, non-image files)
removed = 0
for f in DATA_DIR.rglob("*"):
    if not f.is_file():
        continue
    ok = f.suffix.lower() in IMG_EXTS
    if ok:
        try:
            tf.io.decode_image(tf.io.read_file(str(f)), channels=3, expand_animations=False)
        except Exception:
            ok = False
    if not ok:
        f.unlink()
        removed += 1

print(f"Removed {removed} unreadable/non-image files")
print("Images per class after cleaning:", count_images(DATA_DIR))

## 3. Train / validation / test split (70 / 15 / 15)
We split the **file list** per class (stratified, seeded) into `train/`, `val/` and `test/` folders, then load each with
`image_dataset_from_directory`. Doing the split at file level guarantees the test set never overlaps with training or validation.

In [ ]:
SPLIT_DIR = DATA_DIR.parent / "split"
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

rng = random.Random(SEED)
for class_dir in sorted(d for d in DATA_DIR.iterdir() if d.is_dir()):
    files = sorted(f for f in class_dir.iterdir() if f.is_file())
    rng.shuffle(files)
    n_train, n_val = int(0.70 * len(files)), int(0.15 * len(files))
    parts = {"train": files[:n_train], "val": files[n_train:n_train + n_val], "test": files[n_train + n_val:]}
    for split, split_files in parts.items():
        out = SPLIT_DIR / split / class_dir.name
        out.mkdir(parents=True, exist_ok=True)
        for f in split_files:
            shutil.copy2(f, out / f.name)

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode="int", shuffle=shuffle, seed=SEED)

train_raw = load("train", shuffle=True)
val_raw   = load("val",   shuffle=False)
test_raw  = load("test",  shuffle=False)

class_names = train_raw.class_names
NUM_CLASSES = len(class_names)
print("Classes (in order):", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.cache().prefetch(AUTOTUNE)
val_ds   = val_raw.cache().prefetch(AUTOTUNE)
test_ds  = test_raw.cache().prefetch(AUTOTUNE)

In [ ]:
# Grid of sample training images
plt.figure(figsize=(12, 7))
for images, labels in train_raw.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], color=INK)
        plt.axis("off")
plt.suptitle("Sample training images", color=INK)
plt.tight_layout(); plt.show()

## 4. Data augmentation
Random flips, rotations, zoom, contrast and brightness create new variations of the training images and reduce overfitting.
Augmentation is applied **only to the training pipeline**, so the saved models stay simple to load in the web app.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name="data_augmentation")

train_aug_ds = (train_raw
                .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
                .prefetch(AUTOTUNE))

# Preview: one image, nine random augmentations
plt.figure(figsize=(8, 8))
for images, _ in train_raw.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        aug = data_augmentation(images[:1], training=True)
        plt.imshow(np.clip(aug[0].numpy(), 0, 255).astype("uint8"))
        plt.axis("off")
plt.suptitle("Augmentation examples", color=INK)
plt.tight_layout(); plt.show()

## 5. Model A — Custom CNN from scratch
4 convolution blocks (Conv → BatchNorm → ReLU → MaxPool), then GlobalAveragePooling, Dropout and a softmax output.
The model takes raw 0–255 pixels and rescales them itself. The last conv layer is named `last_conv` — Grad-CAM uses it.

In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = layers.Rescaling(1.0 / 255)(inputs)
    for i, filters in enumerate([32, 64, 128, 256]):
        name = "last_conv" if i == 3 else f"conv_{i + 1}"
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=name)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")

custom_cnn = build_custom_cnn()
custom_cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
custom_cnn.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]
history_cnn = custom_cnn.fit(train_aug_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)

## 6. Model B — MobileNetV2 transfer learning
**Phase 1:** use MobileNetV2 pre-trained on ImageNet as a frozen feature extractor and train only a new classifier head.
**Phase 2 (fine-tuning):** unfreeze the top ~30 layers and continue training with a very low learning rate.

In [ ]:
base_model = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = layers.Rescaling(1.0 / 127.5, offset=-1)(inputs)   # same as mobilenet_v2.preprocess_input: [0,255] -> [-1,1]
x = base_model(x, training=False)                      # keep BatchNorm in inference mode, also while fine-tuning
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
mobilenet = keras.Model(inputs, outputs, name="mobilenetv2_transfer")

mobilenet.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

INITIAL_EPOCHS = 15
history_tl = mobilenet.fit(train_aug_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS,
                           callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)])

In [ ]:
# Fine-tuning: unfreeze only the top ~30 layers of the base model
FINE_TUNE_LAYERS = 30
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
print(f"Trainable layers in base: {sum(l.trainable for l in base_model.layers)} / {len(base_model.layers)}")

mobilenet.compile(optimizer=keras.optimizers.Adam(1e-5),   # low LR so we don't destroy pre-trained features
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

fine_tune_start = len(history_tl.history["loss"])
history_ft = mobilenet.fit(train_aug_ds, validation_data=val_ds,
                           epochs=fine_tune_start + 15, initial_epoch=fine_tune_start,
                           callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])

## 7. Training curves
Accuracy and loss per epoch for both models. The dashed line marks where MobileNetV2 fine-tuning starts.

In [ ]:
def merge(*histories):
    out = {}
    for h in histories:
        for k, v in h.history.items():
            out.setdefault(k, []).extend(v)
    return out

curves = {"Custom CNN": (history_cnn.history, None),
          "MobileNetV2": (merge(history_tl, history_ft), fine_tune_start)}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (name, (h, ft)) in enumerate(curves.items()):
    for col, metric in enumerate(["accuracy", "loss"]):
        ax = axes[row, col]
        epochs = range(1, len(h[metric]) + 1)
        ax.plot(epochs, h[metric], color=C1, lw=2, label="train")
        ax.plot(epochs, h["val_" + metric], color=C2, lw=2, label="validation")
        if ft:
            ax.axvline(ft + 0.5, color=MUTED, ls="--", lw=1)
            ax.text(ft + 0.7, ax.get_ylim()[1], "fine-tune", color=MUTED, va="top", fontsize=9)
        ax.set_title(f"{name} — {metric}", color=INK)
        ax.set_xlabel("epoch"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 8. Evaluation on the test set
Accuracy, classification report and confusion matrix for each model, then a side-by-side comparison.

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])

results = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, model) in zip(axes, [("Custom CNN", custom_cnn), ("MobileNetV2", mobilenet)]):
    probs = model.predict(test_ds, verbose=0)
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    results[name] = {"accuracy": acc, "macro F1": report["macro avg"]["f1-score"]}

    print(f"===== {name} =====  test accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} — confusion matrix", color=INK); ax.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
# Bar chart comparing the two models (accuracy and macro F1 on the test set)
metrics = ["accuracy", "macro F1"]
x = np.arange(len(metrics)); w = 0.36
fig, ax = plt.subplots(figsize=(7, 4.5))
for i, (name, color) in enumerate([("Custom CNN", C1), ("MobileNetV2", C2)]):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + (i - 0.5) * w, vals, w * 0.94, color=color, label=name)
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in vals], padding=3, color=INK, fontsize=10)
ax.set_xticks(x, metrics); ax.set_ylim(0, 1.08); ax.set_ylabel("score")
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.grid(axis="x", visible=False)
ax.set_title("Test-set performance: Custom CNN vs MobileNetV2", color=INK)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

## 9. Grad-CAM — where did the model look?
Grad-CAM uses the gradients of the predicted class with respect to the **custom CNN's last conv layer** (`last_conv`)
to build a heatmap. Red/yellow = regions that most influenced the decision.

In [ ]:
LAST_CONV = "last_conv"

def make_gradcam_heatmap(img_batch, model, last_conv_name=LAST_CONV, pred_index=None):
    """Return a (h, w) heatmap in [0, 1] for one image batch of shape (1, H, W, 3)."""
    grad_model = keras.Model(model.inputs, [model.get_layer(last_conv_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch, training=False)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        class_score = preds[:, pred_index]
    grads = tape.gradient(class_score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))              # importance of each feature map
    heatmap = tf.reduce_sum(conv_out[0] * weights, axis=-1)      # weighted sum of feature maps
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(img, heatmap, alpha=0.4):
    """Blend a jet-coloured heatmap over an RGB uint8 image."""
    heat = tf.image.resize(heatmap[..., None], img.shape[:2]).numpy()[..., 0]
    colored = matplotlib.colormaps["jet"](heat)[..., :3] * 255
    return np.clip(img * (1 - alpha) + colored * alpha, 0, 255).astype("uint8")

images, labels = next(iter(test_raw.unbatch().shuffle(200, seed=SEED).batch(6)))
plt.figure(figsize=(15, 5.5))
for i in range(len(images)):
    img = images[i].numpy()
    probs = custom_cnn.predict(img[None], verbose=0)[0]
    heatmap = make_gradcam_heatmap(img[None], custom_cnn)
    plt.subplot(2, 6, i + 1); plt.imshow(img.astype("uint8")); plt.axis("off")
    plt.title(f"true: {class_names[labels[i]]}", color=INK, fontsize=10)
    plt.subplot(2, 6, i + 7); plt.imshow(overlay_heatmap(img, heatmap)); plt.axis("off")
    plt.title(f"pred: {class_names[probs.argmax()]} ({probs.max():.0%})", color=INK, fontsize=10)
plt.suptitle("Grad-CAM (custom CNN, last conv layer)", color=INK)
plt.tight_layout(); plt.show()

## 10. Predict on your own image (optional)
Set `UPLOAD_IMAGE = True` and re-run this cell to upload a road photo.
By default it's `False` so **Run all never stops to wait for input** — it predicts on a random test image instead.

In [ ]:
UPLOAD_IMAGE = False   # ← change to True to upload your own photo

def predict_and_explain(img_uint8, title=""):
    batch = img_uint8.astype("float32")[None]
    p_mob = mobilenet.predict(batch, verbose=0)[0]
    heatmap = make_gradcam_heatmap(batch, custom_cnn, pred_index=int(p_mob.argmax()))
    print(title)
    for name, p in zip(class_names, p_mob):
        print(f"  {name:>10}: {p:.2%}")
    plt.figure(figsize=(9, 4.5))
    plt.subplot(1, 2, 1); plt.imshow(img_uint8); plt.axis("off")
    plt.title(f"MobileNetV2: {class_names[p_mob.argmax()]} ({p_mob.max():.1%})", color=INK)
    plt.subplot(1, 2, 2); plt.imshow(overlay_heatmap(img_uint8, heatmap)); plt.axis("off")
    plt.title("Grad-CAM (custom CNN)", color=INK)
    plt.tight_layout(); plt.show()

def load_image(path):
    img = keras.utils.load_img(path, target_size=IMG_SIZE)
    return np.array(img, dtype="uint8")

if UPLOAD_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        predict_and_explain(load_image(fname), title=f"Uploaded: {fname}")
else:
    sample = random.Random().choice(sorted((SPLIT_DIR / "test").rglob("*.*")))
    predict_and_explain(load_image(sample), title=f"Demo on test image: {sample.parent.name}/{sample.name}")

## 11. Save the models and class names
Saves both models in the native Keras format plus `classes.json` (class names in index order), then downloads all three files.
Put them in the repo's **`models/`** folder. Note the TensorFlow version printed below — install the same (or a close) version locally.

In [ ]:
SAVE_DIR = pathlib.Path("/content/models") if os.path.exists("/content") else pathlib.Path("models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

mobilenet_path = SAVE_DIR / "road_surface_mobilenetv2.keras"
custom_path    = SAVE_DIR / "road_surface_custom_cnn.keras"
classes_path   = SAVE_DIR / "classes.json"

mobilenet.save(mobilenet_path)
custom_cnn.save(custom_path)
with open(classes_path, "w") as f:
    json.dump(class_names, f)

for p in [mobilenet_path, custom_path, classes_path]:
    print(f"Saved {p}  ({p.stat().st_size / 1e6:.1f} MB)")
print("Classes:", class_names)
print("TensorFlow version used for training:", tf.__version__)
print("Keras version:", keras.__version__)

try:
    from google.colab import files
    for p in [mobilenet_path, custom_path, classes_path]:
        files.download(str(p))
    print("Downloads triggered — if your browser asks, allow multiple downloads.")
except ImportError:
    print("Not running in Colab — files are saved in", SAVE_DIR.resolve())